# APTOS 2019 on Kaggle, part 2 of 2: submission
### Grades the test set with the 50 models from `kaggle_train.ipynb` and writes `submission.csv`

This notebook trains nothing. It loads the snapshot weights and the cut-offs saved by the training notebook, pre-processes each test image the same way (CLAHE + blur, 300 px), averages the regression output of every model, and turns the mean into a grade with the OOF-tuned cut-offs.

**How to submit**

1. Run `kaggle_train.ipynb` as a committed version first (Save & Run All) and wait until it has finished.
2. **Code -> New Notebook -> File -> Import Notebook**, upload this file.
3. Sidebar **Add Input**: **Competitions -> APTOS 2019 Blindness Detection**, and **Your Work -> Notebooks ->** the training notebook (its output, with `meta.json` and `models/`).
4. Sidebar **Settings: Accelerator = GPU**, **Internet = Off** (Kaggle rejects code-competition submissions with internet on).
5. **Save Version -> Save & Run All**. Check the log: it prints the grade distribution and writes `/kaggle/working/submission.csv` (`id_code,diagnosis`).
6. Competition page -> **Late Submission** (or **Submit Prediction**) -> choose this notebook, the version, and `submission.csv`. Kaggle re-runs it on the full hidden test set (~13,000 images), which takes ~20-30 minutes, and shows the public and private scores. The competition has closed, so the score is not ranked on the leaderboard.

Nothing here downloads anything, so it runs with internet off: `timm` is preinstalled on Kaggle, and the network is built with `pretrained=False` before the saved weights are loaded.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from tqdm.auto import tqdm

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "PDR"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

ON_KAGGLE = Path("/kaggle/working").exists()
OUTPUT_DIR = Path(os.environ.get("DR_OUTPUT_DIR", "/kaggle/working" if ON_KAGGLE else "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU found (very slow!)")
print("timm", timm.__version__, "| torch", torch.__version__)

## 1. Find the competition data and the trained models

In [ ]:
def find_input_dir(*markers, env=None):
    # The folder under /kaggle/input holding all `markers`. Kaggle's mount layout differs between UI versions.
    if env and os.environ.get(env):
        return Path(os.environ[env])
    for pattern in ("*", "*/*", "*/*/*", "*/*/*/*", "*/*/*/*/*"):
        for folder in Path("/kaggle/input").glob(pattern):
            if all((folder / m).exists() for m in markers):
                return folder
    raise FileNotFoundError(f"no input folder with {markers}. Check the inputs in the notebook sidebar.")


COMP_DIR = find_input_dir("test.csv", "test_images", env="APTOS_DIR")
WEIGHTS_DIR = find_input_dir("meta.json", "models", env="DR_MODEL_DIR")
meta = json.loads((WEIGHTS_DIR / "meta.json").read_text())
cfg = meta["config"]
cutoffs = np.array(meta["cutoffs"])
print(f"competition data: {COMP_DIR}\nmodels:           {WEIGHTS_DIR}")
print(f"{len(meta['models'])} models from folds {meta['folds']}, {cfg['backbone']} at {cfg['image_size']} px, "
      f"cut-offs {cutoffs.tolist()}")
print({k: v for k, v in meta.items() if k.startswith("oof_")})
if meta["quick_test"]:
    print("\nWARNING: these models come from a QUICK_TEST run of the training notebook. Do not submit this.")
if meta["timm_version"] != timm.__version__:
    print(f"note: trained with timm {meta['timm_version']}, running {timm.__version__}")

## 2. Load the models

In [ ]:
class DRNet(nn.Module):
    # EfficientNet + the paper's head, ending in one output: the grade as a number on the 0-4 scale.
    def __init__(self, backbone="efficientnet_b3", pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained, num_classes=0, global_pool="avg")
        self.head = nn.Sequential(
            nn.Linear(self.backbone.num_features, 512), nn.Dropout(0.5), nn.ReLU(inplace=True),
            nn.Linear(512, 512), nn.Dropout(0.25), nn.ReLU(inplace=True),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))[:, 0]


_mean = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
_std = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
_black = -_mean / _std     # what a black pixel looks like after normalisation


def to_model_input(batch_uint8):
    # numpy (B, H, W, 3) uint8 -> normalised float tensor (B, 3, H, W) on the GPU.
    x = torch.from_numpy(batch_uint8).to(device).permute(0, 3, 1, 2).float() / 255.0
    return ((x - _mean) / _std).contiguous(memory_format=torch.channels_last)


models = []
for name in tqdm(meta["models"], desc="loading models"):
    model = DRNet(cfg["backbone"], pretrained=False)
    model.load_state_dict(torch.load(WEIGHTS_DIR / "models" / name, map_location="cpu"))
    models.append(model.to(device).to(memory_format=torch.channels_last).eval())
use_amp = device.type == "cuda"

## 3. Grade the test set

Images are pre-processed in chunks with all CPU cores, then every model scores every image (mixed precision on the GPU). An image that cannot be read gets a black picture instead of stopping the run: a crash during Kaggle's hidden re-run would give no score at all.

In [ ]:
def preprocess(path, size):
    # Load -> shrink to <= 512 px -> CLAHE + blur -> square resize. Returns RGB uint8 (size, size, 3).
    # Identical in the training and the submission notebook.
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    h, w = bgr.shape[:2]
    if max(h, w) > 512:
        scale = 512 / max(h, w)
        bgr = cv2.resize(bgr, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    bgr = cv2.GaussianBlur(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), (5, 5), 0)
    bgr = cv2.resize(bgr, (size, size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def safe_preprocess(path):
    try:
        return preprocess(path, cfg["image_size"])
    except Exception as err:                     # never let one bad file sink the whole submission
        print(f"warning: {path.name}: {err!r}, using a black image")
        return np.zeros((cfg["image_size"], cfg["image_size"], 3), dtype=np.uint8)


@torch.no_grad()
def ensemble_score(batch_uint8, batch_size=64):
    # Mean regression output of all models, one number per image.
    total = torch.zeros(len(batch_uint8), device=device)
    for start in range(0, len(batch_uint8), batch_size):
        x = to_model_input(batch_uint8[start:start + batch_size])
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            for model in models:
                total[start:start + len(x)] += model(x).float()
    return (total / len(models)).cpu().numpy()


test = pd.read_csv(COMP_DIR / "test.csv")
paths = [COMP_DIR / "test_images" / f"{i}.png" for i in test["id_code"]]
scores = np.empty(len(test))
CHUNK = 512
t0 = time.time()
with ThreadPoolExecutor(os.cpu_count()) as pool:            # OpenCV releases the GIL, so threads are enough
    for start in tqdm(range(0, len(paths), CHUNK), desc="test chunks"):
        batch = np.stack(list(pool.map(safe_preprocess, paths[start:start + CHUNK])))
        scores[start:start + len(batch)] = ensemble_score(batch)
print(f"{len(test)} test images, {time.time() - t0:.0f}s")

## 4. Write `submission.csv`

In [ ]:
submission = pd.DataFrame({"id_code": test["id_code"], "diagnosis": np.digitize(scores, cutoffs).astype(int)})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
assert len(submission) == len(test) and submission["diagnosis"].between(0, NUM_CLASSES - 1).all()

counts = submission["diagnosis"].value_counts().reindex(range(NUM_CLASSES), fill_value=0)
print(pd.DataFrame({"grade": CLASS_NAMES, "images": counts.values, "share": (counts / len(submission)).round(3).values}).to_string(index=False))
print(f"\nwrote {OUTPUT_DIR / 'submission.csv'}")
submission.head()